# Topic 11 - Type Hints and Pydantic Models

This notebook practices type hints and Pydantic validation by modeling real LLM API payloads. No SDKs — just typed data structures that enforce contracts at the boundary.

**Rule:** write all code by hand. No Copilot. Run mypy and pytest yourself.

## Learning objectives

By the end, you should be able to:

- write type annotations for primitives, Optionals, Unions, and containers
- define Pydantic models with field constraints
- parse and validate external data with `model_validate`
- serialize models to dict/JSON with `model_dump`
- explain why validation belongs at the boundary, not scattered through code

## 1. Type annotation basics

Modern Python (3.9+) uses built-in generics. No `from typing import List, Dict` needed for standard containers.

In [1]:
# Primitive annotations
name: str = "invoice"
count: int = 5
ratio: float = 1.2
active: bool = True
nothing: None = None

# Container annotations (Python 3.9+)
ids: list[int] = [1, 2, 3]
tags: set[str] = {"a", "b"}
mapping: dict[str, int] = {"a": 1, "b": 2}

# Optional and Union
from typing import Optional, Union

name_opt: Optional[str] = None  # str | None
value: Union[int, float] = 42   # int | float

print("Basic annotations assigned successfully")

Basic annotations assigned successfully


### Exercise: annotate this function

Add type hints to `calculate_total`. What does it accept? What does it return?

In [2]:
def calculate_total(items):
    total = 0.0
    for item in items:
        total += item["quantity"] * item["unit_price"]
    return total

# Test it
sample = [
    {"description": "Widget", "quantity": 3, "unit_price": 10.0},
    {"description": "Gadget", "quantity": 2, "unit_price": 25.5},
]
print(f"Total: {calculate_total(sample)}")

Total: 81.0


## 2. Pydantic BaseModel

Pydantic models are classes that validate data at construction time.

In [3]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional

class LineItem(BaseModel):
    description: str
    quantity: int = Field(gt=0)
    unit_price: float = Field(gt=0)

class Invoice(BaseModel):
    invoice_number: str
    date: str  # expect YYYY-MM-DD
    items: list[LineItem]
    total: float = Field(gt=0)
    vendor: Optional[str] = None

# Valid construction
invoice = Invoice(
    invoice_number="INV-001",
    date="2024-01-15",
    items=[
        LineItem(description="Widget", quantity=3, unit_price=10.0),
        LineItem(description="Gadget", quantity=2, unit_price=25.5),
    ],
    total=81.0,
    vendor="Acme Corp"
)
print("Valid invoice created:")
print(invoice.model_dump_json(indent=2))

Valid invoice created:
{
  "invoice_number": "INV-001",
  "date": "2024-01-15",
  "items": [
    {
      "description": "Widget",
      "quantity": 3,
      "unit_price": 10.0
    },
    {
      "description": "Gadget",
      "quantity": 2,
      "unit_price": 25.5
    }
  ],
  "total": 81.0,
  "vendor": "Acme Corp"
}


### Validation errors

Invalid data raises `ValidationError` with structured details.

In [4]:
# Invalid: negative quantity
try:
    bad = LineItem(description="Widget", quantity=-1, unit_price=10.0)
except ValidationError as e:
    print("ValidationError caught:")
    for error in e.errors():
        print(f"  Field: {error['loc']}, Type: {error['type']}, Msg: {error['msg']}")

# Invalid: missing required field
try:
    bad = Invoice(invoice_number="INV-002", date="2024-01-15", items=[], total=100)
except ValidationError as e:
    print("\nMissing field error:")
    for error in e.errors():
        print(f"  Field: {error['loc']}, Type: {error['type']}")

ValidationError caught:
  Field: ('quantity',), Type: greater_than, Msg: Input should be greater than 0


## 3. Serialization: model → dict / JSON

Pydantic models convert to dict or JSON for transport.

In [5]:
# model_dump() → dict
as_dict = invoice.model_dump()
print("As dict:")
print(f"  Type: {type(as_dict)}")
print(f"  Keys: {list(as_dict.keys())}")

# model_dump_json() → JSON string
as_json = invoice.model_dump_json(indent=2)
print("\nAs JSON:")
print(as_json)

# Exclude None values
print("\nExclude None:")
print(invoice.model_dump(exclude_none=True))

As dict:
  Type: <class 'dict'>
  Keys: ['invoice_number', 'date', 'items', 'total', 'vendor']

As JSON:
{
  "invoice_number": "INV-001",
  "date": "2024-01-15",
  "items": [
    {
      "description": "Widget",
      "quantity": 3,
      "unit_price": 10.0
    },
    {
      "description": "Gadget",
      "quantity": 2,
      "unit_price": 25.5
    }
  ],
  "total": 81.0,
  "vendor": "Acme Corp"
}

Exclude None:
{'invoice_number': 'INV-001', 'date': '2024-01-15', 'items': [{'description': 'Widget', 'quantity': 3, 'unit_price': 10.0}, {'description': 'Gadget', 'quantity': 2, 'unit_price': 25.5}], 'total': 81.0, 'vendor': 'Acme Corp'}


## 4. Parsing: dict/JSON → model

`model_validate` and `model_validate_json` parse external data and validate it.

In [6]:
# From dict
raw_data = {
    "invoice_number": "INV-003",
    "date": "2024-02-20",
    "items": [
        {"description": "Service", "quantity": 1, "unit_price": 500.0}
    ],
        "total": 500.0,
    "vendor": "Service Co"
}

parsed = Invoice.model_validate(raw_data)
print("Parsed from dict:")
print(f"  invoice_number: {parsed.invoice_number}")
print(f"  items count: {len(parsed.items)}")

# From JSON string
json_str = '{"invoice_number": "INV-004", "date": "2024-03-10", "items": [{"description": "Part", "quantity": 5, "unit_price": 2.0}], "total": 10.0}'
parsed2 = Invoice.model_validate_json(json_str)
print(f"\nParsed from JSON: {parsed2.invoice_number}")

Parsed from dict:
  invoice_number: INV-003
  items count: 1

Parsed from JSON: INV-004


## 5. LLM API payload modeling

Model the request/response shapes you'll use in Topics 12, 17, and beyond.

In [7]:
from typing import Literal

# Chat completion request (OpenAI-compatible)
class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant", "tool"]
    content: str
    # tool_calls, tool_call_id omitted for simplicity

class ChatCompletionRequest(BaseModel):
    model: str
    messages: list[ChatMessage]
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=256, gt=0)
    stream: bool = False

# Chat completion response
class Choice(BaseModel):
    index: int
    message: ChatMessage
    finish_reason: Optional[str] = None

class Usage(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

class ChatCompletionResponse(BaseModel):
    id: str
    object: Literal["chat.completion"]
    created: int
    model: str
    choices: list[Choice]
    usage: Usage

# Test request
req = ChatCompletionRequest(
    model="gpt-4o-mini",
    messages=[
        ChatMessage(role="system", content="You are helpful."),
        ChatMessage(role="user", content="Hello")
    ],
    temperature=0.2,
    max_tokens=128
)
print("Request JSON:")
print(req.model_dump_json(indent=2))

Request JSON:
{
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "system",
      "content": "You are helpful."
    },
    {
      "role": "user",
      "content": "Hello"
    }
  ],
  "temperature": 0.2,
  "max_tokens": 128,
  "stream": false
}


## 6. Boundary validation pattern

The pattern: parse once at the boundary, use typed models internally.

In [8]:
# Simulated external API response (as raw dict)
raw_response = {
    "id": "chatcmpl-abc123",
    "object": "chat.completion",
    "created": 1700000000,
    "model": "gpt-4o-mini",
    "choices": [
        {
            "index": 0,
            "message": {"role": "assistant", "content": "Hello! How can I help?"},
            "finish_reason": "stop"
        }
    ],
    "usage": {"prompt_tokens": 15, "completion_tokens": 8, "total_tokens": 23}
}

# Parse once at boundary
response = ChatCompletionResponse.model_validate(raw_response)

# Internal code uses typed model
def extract_text(resp: ChatCompletionResponse) -> str:
    """Extract assistant text from validated response."""
    return resp.choices[0].message.content

def calculate_cost(resp: ChatCompletionResponse, price_in: float, price_out: float) -> float:
    """Cost in USD from usage."""
    return (resp.usage.prompt_tokens / 1_000_000 * price_in +
            resp.usage.completion_tokens / 1_000_000 * price_out)

print(f"Text: {extract_text(response)}")
print(f"Cost: ${calculate_cost(response, 0.15, 0.60):.6f}")

Text: Hello! How can I help?
Cost: $0.000007


## 7. Configuration: forbid extra fields

Strict models reject unknown fields — catches API drift early.

In [9]:
from pydantic import ConfigDict

class StrictResponse(BaseModel):
    model_config = ConfigDict(extra='forbid')
    id: str
    content: str

# This works
ok = StrictResponse(id="1", content="hello")
print(f"Valid: {ok.id}")

# This fails - extra field not allowed
try:
    bad = StrictResponse(id="1", content="hello", unexpected_field="bad")
except ValidationError as e:
    print(f"Extra field rejected: {e.errors()[0]['type']}")

Valid: 1
Extra field rejected: extra_forbidden


## 8. Completion exercises

Do these by hand. No autocomplete.

In [ ]:
# EXERCISE 1: a validator that enforces the YYYY-MM-DD date format
import re

from pydantic import BaseModel, Field, field_validator


class DatedRecord(BaseModel):
    date: str

    @field_validator("date")
    @classmethod
    def validate_date_format(cls, v: str) -> str:
        if not re.match(r"^\d{4}-\d{2}-\d{2}$", v):
            raise ValueError("Date must be in YYYY-MM-DD format")
        return v


print(DatedRecord(date="2024-01-15"))
try:
    DatedRecord(date="15/01/2024")
except ValueError as exc:
    print("rejected:", exc)

# EXERCISE 2: a ToolCall model for function calling (Topic 17)
from typing import Any, Dict, Literal


class ToolCall(BaseModel):
    type: Literal["function"] = "function"
    name: str
    arguments: Dict[str, Any]


tool_call = ToolCall(name="my_function", arguments={"arg1": "value1"})
print(tool_call.model_dump_json(indent=2))

# EXERCISE 3: a model for the Mini-Project A summariser config
class SummariserConfig(BaseModel):
    style: Literal["paragraph", "bullets", "tldr"]
    max_tokens: int = Field(gt=0, le=4096)
    temperature: float = Field(ge=0.0, le=2.0)
    price_in: float
    price_out: float


config = SummariserConfig(
    style="tldr", max_tokens=256, temperature=0.3, price_in=0.15, price_out=0.60
)
print(config.model_dump_json(indent=2))

print("Exercises ready - solutions written by hand.")


Exercises ready — write solutions by hand.
